# Геолокация снимков · итоговый эксперимент

Сравниваем усиленную нейросеть, поиск по эмбеддингам, фиксированные гибриды, CatBoost поверх прогнозов и классификацию географических ячеек. Цель — минимальный RMSE геодезических расстояний. MAE показывает среднее расстояние ошибки. Обе метрики считаются в километрах по эллипсоиду WGS84.

![Схема подготовки, вложенной кросс-валидации, переобучения и финального теста](docs/workflow.svg)
**Результат прогона на T4:** победитель по CV — 512 ячеек, взвешенный прогноз по вероятностям (RMSE 1 098,5 км). После обучения на всех 85% MAE на отложенных снимках составил 319,1 км, RMSE — 1 065,6 км. Цифры повторного запуска могут немного меняться.


## 1. Среда и данные

В Colab выберите GPU. Если код уже находится в `/content/train250k-geo-ml`, достаточно указать путь к датасету на Диске. Для нового сеанса загрузите ZIP проекта через панель **Файлы** и задайте `ZIP_PATH`. При локальном запуске держите ноутбук рядом с каталогом `src`.

`GEO_CV_ROOT` хранит разбиение, промежуточные прогнозы и метрики. Для исходного эксперимента используйте тот же каталог, чтобы сохранить состав отложенных 37 500 снимков.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from IPython.display import SVG, display

REPO_DIR = "/content/train250k-geo-ml"  # @param {type:"string"}
ZIP_PATH = ""  # @param {type:"string"}
TRAIN_250K_DIR = "/content/drive/MyDrive/train_250k"  # @param {type:"string"}
GEO_CV_ROOT = "/content/train250k-cv"  # @param {type:"string"}
INSTALL_DEPENDENCIES = True  # @param {type:"boolean"}

checkout = Path(REPO_DIR).expanduser()
here = Path.cwd()
repo = next((p for p in (checkout, here, here.parent) if (p / "src" / "run_v2.py").is_file()), None)
if repo is None:
    if not ZIP_PATH:
        raise FileNotFoundError("Укажите ZIP_PATH или путь к новой версии проекта в REPO_DIR.")
    if checkout.exists():
        raise FileExistsError("Для нового ZIP укажите свободный путь в REPO_DIR.")
    checkout.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=checkout.parent) as tmp:
        with ZipFile(Path(ZIP_PATH).expanduser()) as archive:
            archive.extractall(tmp)
        roots = [Path(tmp), *Path(tmp).iterdir()]
        source = next((p for p in roots if (p / "src" / "run_v2.py").is_file()), None)
        if source is None:
            raise FileNotFoundError("В ZIP отсутствует src/run_v2.py.")
        from shutil import copytree
        copytree(source, checkout)
    repo = checkout

data, work = (Path(p).expanduser().resolve() for p in (TRAIN_250K_DIR, GEO_CV_ROOT))
env = {**os.environ, "TRAIN_250K_DIR": str(data), "GEO_CV_ROOT": str(work)}
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements.txt")], check=True)

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Выберите среду Colab с GPU.")
print(torch.cuda.get_device_name())

def run_src(script, *args):
    subprocess.run([sys.executable, "-u", str(repo / "src" / script), *args], cwd=repo, env=env, check=True)

if (repo / "docs" / "workflow.svg").is_file():
    display(SVG(filename=str(repo / "docs" / "workflow.svg")))

In [ ]:
if not (work / "split.json").is_file():
    if (data.is_relative_to("/content/drive") or work.is_relative_to("/content/drive")) and not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive
        drive.mount("/content/drive")
    run_src("prepare.py")
split = json.loads((work / "split.json").read_text())
assert split["trainval"] == 212_500 and split["test"] == 37_500
assert split["fold_sizes"] == [42_500] * 5
print("Разбиение подтверждено", split["fold_sizes"], "тест", split["test"])

## 2. Выбор конфигурации

На каждом внешнем фолде обучение получает 170 000 снимков, валидация — 42 500. CatBoost учится на прогнозах для строк, исключённых из обучения соответствующих базовых моделей во внутренних разбиениях. Базовые модели и географические ячейки строятся только по внешней обучающей части.

Дополнительные признаки строятся из самих прогнозов и поиска соседей, включая сходство, разброс координат соседей и согласие моделей. Координаты и страна предсказываемого снимка на вход не подаются.

Географическая ветка делит сферу на 512 или 1 024 ячейки. Частоты ячеек дают ориентир без признаков снимка. Нейросеть по двум эмбеддингам оценивает вероятности ячеек. Из вероятностей получаем координату центра наиболее вероятной ячейки или взвешенный прогноз.
Вероятность без признаков: $p_j=n_j/N_{\mathrm{train}}$. Классификатор предсказывает $p_j(x)$ через softmax. Для RMSE сравниваем MAP (центр с максимальной вероятностью) и сферическое среднее Фреше — приближённый минимум ожидаемого квадрата расстояния. Сами оценки расстояния всегда считаются точно по WGS84.


In [ ]:
import pandas as pd

run_src("run_v2.py", "cv")
cv = json.loads((work / "v2" / "cv-summary.json").read_text())
table = (pd.DataFrame(cv["candidates"])
         .loc[:, ["name", "cv_mae_km", "cv_rmse_km"]]
         .rename(columns={"name": "Вариант", "cv_mae_km": "MAE, км", "cv_rmse_km": "RMSE, км"})
         .sort_values("RMSE, км"))
print("Победитель по пяти фолдам", cv["winner"])
display(table.round(1))

## 3. Переобучение и итог

Переобучаем зафиксированного победителя на всех 212 500 снимках. Для стека на этой выборке заново строятся внутренние прогнозы, после чего базовые модели обучаются на всех 85%. Только затем открываем метки отложенных 37 500 снимков и считаем MAE и RMSE.

Этот тест уже использовался для отчёта о предыдущем гибриде. Его результат служит историческим ориентиром. Подбор здесь использует только пять фолдов из 85%.

In [ ]:
run_src("run_v2.py", "final")
result = json.loads((work / "v2" / "results.json").read_text())
assert result["winner"] == cv["winner"]
test = result["test"]
assert test["n"] == 37_500
display(pd.DataFrame([{"Модель": result["winner"], "Снимков": test["n"],
                       "MAE, км": test["mae_km"], "RMSE, км": test["rmse_km"]}]).round(1))
if result.get("historical_reference"):
    print("Исторический ориентир", result["historical_reference"])

In [ ]:
# Для победителя-классификатора покажем вероятности на одном снимке из 85%.
# Этот пример иллюстрирует модель и не используется для выбора гиперпараметров.
checkpoint_path = work / "v2" / "best-cells.pt"
if result["winner"].startswith("cell_") and checkpoint_path.is_file():
    import numpy as np
    import matplotlib.pyplot as plt
    sys.path.insert(0, str(repo / "src"))
    from cell_v2 import CellClassifier
    from experiment import decode

    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model = CellClassifier(1280, ckpt["k"]).to("cuda")
    model.load_state_dict(ckpt["state_dict"]); model.eval()
    x = np.load(work / "dev_x.npy", mmap_mode="r")[:1].copy()
    with torch.inference_mode():
        logits = model(torch.tensor(x, device="cuda") * ckpt["input_scale"])[0].cpu()
        posterior = logits.masked_fill(ckpt["cluster_counts"] == 0, -torch.inf).softmax(dim=0)
    lat, lon = decode(ckpt["centers"].numpy())
    prior = ckpt["prior_probs"].numpy()
    prob = posterior.numpy()
    top = np.argsort(prob)[-8:][::-1]

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.7), layout="constrained")
    axes[0].scatter(lon, lat, c=prob, s=8 + 420 * prob, cmap="viridis")
    axes[0].set(xlim=(-180,180), ylim=(-90,90), title="Вероятности географических ячеек",
                xlabel="Долгота, °", ylabel="Широта, °")
    axes[1].bar(np.arange(len(top))-.2, prob[top], .4, label="По эмбеддингам")
    axes[1].bar(np.arange(len(top))+.2, prior[top], .4, label="По частотам")
    axes[1].set(xticks=np.arange(len(top)), xticklabels=top, xlabel="ID ячейки",
                ylabel="Вероятность", title="Восемь наиболее вероятных ячеек")
    axes[1].legend()
    plt.show()
